# 08 · The grid (stage S7)

Every item at every kept fraction `k` and loop count `R`, with the compressor chosen in S5, controller features on.
The controller later learns and is evaluated from this table (`grid.parquet`) without running the model again.
Items: `ctrl_train` + `dev` (the test split stays sealed until S12). A sample of `n_rerun` items runs a second time
(`rerun_id = 1`) to measure how much the results move on their own.

**How to run:** Run All, no edits: one item, the grid batches and batch 1 against batch 8, a 20-item sample, then the
full run (`RUN_FULL`; long and resumable). `compressor` must be frozen in `configs/mvp.yaml` for the full run.
The grid generates in batches of `grid_batch.size` and writes to `grid_b8/`; latency and every other notebook stay batch 1.
Steps 5-7 (figures, noise rate, headroom) read the saved `grid.parquet` in seconds: header cells and Settings, then those.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import full_gpu_overrides, select_for_notebook

CONFIG = "configs/mvp.yaml"
ALL_GPUS = True  # this notebook: 4 model copies on GPUs 0-7 (big run); False: 0-3 with the config's n_copies
GPU_CFG = full_gpu_overrides(CONFIG) if ALL_GPUS else {}  # unhashed placement keys
GPU_IDS, N_COPIES = select_for_notebook(CONFIG, GPU_CFG)

In [ ]:
import math
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook(CONFIG, overrides=GPU_CFG)
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")

In [ ]:
import pandas as pd

In [ ]:
# Settings
RUN_FULL = True  # full grid (ctrl_train + dev) + noise re-run, 4 copies; stops earlier if a sample check fails
RUN_B1_VS_B8 = True  # Step-2b: batch 1 against the grid batch on dev items (resumable)
RUN_BATCH_CHECK = False  # exact batched = batch 1 check; they do differ (nb07 Nov 26: 13/128), Step-2b measures how much
N_SAMPLE = cfg["sample_items"]  # step 3: dev items, half GQA, half TextVQA
FROZEN = cfg["compressor"] != "none"
grid_cfg = cfg if FROZEN else {**cfg, "compressor": "text_mean"}  # S5 froze divprune; text_mean only if mvp.yaml says none
GRID_BATCH = cfg["grid_batch"]["size"]  # generation batch of every grid cell; vision and compression run per item
CHECK = cfg["grid_batch"]["check"]
OUT_DIR = api.grid_dir(grid_cfg, ROOT)  # grid_b8: rows never mix with batch-1 rows
SAMPLE_DIR = api.grid_dir(grid_cfg, ROOT, "grid_sample")
CELLS = [(k, R) for k in cfg["k_grid"] for R in cfg["R_grid"]]
print("compressor", grid_cfg["compressor"], "(frozen)" if FROZEN else "(NOT frozen: sample only)", "| cells", len(CELLS), "| batch", GRID_BATCH, "|", OUT_DIR.name)
cfg["grid"], cfg["n_rerun"]

### Step-1:- Load the model

In [ ]:
print("[STAGE] load model")
vlms = api.load_models(cfg, N_COPIES)
vlm = vlms[0]

### Step-2:- One item per benchmark, every cell (paste this output)

Raw output, parsed answer and references at R = 3 for each k, then the row of every cell and its features.

In [ ]:
one_items = [api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0], api.load_textvqa(cfg, "dev", limit=1, root=ROOT)[0]]
print("[STAGE] one item per benchmark")
one = {}
for it in one_items:
    rows, features = api.run_item_cells(vlm, it, grid_cfg)
    one[it.benchmark] = (rows, features)
    for row in rows:
        if row["R"] == 3:
            print(f"{it.benchmark} | k={row['k']} R=3 | kept {row['n_vis']}/{row['n_vis_full']}")
            print(f"    raw {row['raw_output']!r} | pred {row['pred']!r} | gold {it.answers[:3]}")
pd.DataFrame([r for rows, _ in one.values() for r in rows])[
    ["benchmark", "k", "R", "n_vis", "n_input", "pred", "correct", "ttft_s", "generate_s"]]

In [ ]:
for benchmark, (rows, features) in one.items():
    assert sorted((r["k"], r["R"]) for r in rows) == sorted(CELLS) == sorted(features), benchmark
    assert all(r["parse_ok"] for r in rows), f"{benchmark}: an answer did not parse"
    assert all(torch.isfinite(f["h_last"]).all() for f in features.values())
print(f"✓ one item per benchmark: {len(CELLS)} cells each, answers parse, features finite")

#### Batched against batch 1, with features (off)

An exact comparison, kept for debugging (`RUN_BATCH_CHECK`): 16 dev items per benchmark at k = 25% and 100%, batched and
each alone. BF16 batch kernels flip near-ties, so some answers differ (nb07 check, Nov 26: 13 of 128); it reports them
and does not stop the run. Features come from floating-point states and are compared with a tolerance.

In [ ]:
if RUN_BATCH_CHECK:
    CHECK_CFG = {**grid_cfg, "k_grid": [0.25, 1.0], "R_grid": [1, 2, 3]}  # for this check only
    N_BATCH_CHECK = 16  # items per benchmark
    CHECK_BATCH = 8  # batch size under test, whatever eval_batch_size is
    check_items = []
    for load in (api.load_gqa, api.load_textvqa):
        check_items += api.batchable_items(vlm, load(cfg, "dev", root=ROOT), CHECK_CFG, N_BATCH_CHECK)
    print("[STAGE] equal-length batches against batch 1")
    compare_rows = []
    for chunk in api.item_chunks(vlm, check_items, CHECK_CFG, CHECK_BATCH):
        together = api.run_items_cells(vlm, chunk, CHECK_CFG)
        alone = [api.run_item_cells(vlm, it, CHECK_CFG) for it in chunk]
        for it, (rows_b, feats_b), (rows_a, feats_a) in zip(chunk, together, alone):
            for row_b, row_a in zip(rows_b, rows_a):
                key = row_a["k"], row_a["R"]
                fa, fb = feats_a[key], feats_b[key]
                compare_rows.append({
                    "item_id": it.item_id, "batch_n": len(chunk), "k": row_a["k"], "R": row_a["R"],
                    "same_output": row_a["raw_output"] == row_b["raw_output"], "same_pred": row_a["pred"] == row_b["pred"],
                    "h_cosine": torch.cosine_similarity(fa["h_last"].float(), fb["h_last"].float(), dim=0).item(),
                    "entropy_diff": abs(fa["entropy"] - fb["entropy"]),
                    "vis_mass_diff": abs(fa["vis_attn_mass"] - fb["vis_attn_mass"])})
    batch_check = pd.DataFrame(compare_rows)
    batch_check[["batch_n", "same_output", "same_pred", "h_cosine", "entropy_diff", "vis_mass_diff"]].agg(["min", "max"])
else:
    print("skipped: RUN_BATCH_CHECK is False")


In [ ]:
if RUN_BATCH_CHECK:
    batched_rows = batch_check[batch_check.batch_n > 1]
    assert len(batched_rows) >= len(batch_check) // 2, "too few rows were really batched to prove anything"
    different = batched_rows[~(batched_rows.same_output & batched_rows.same_pred)]
    print(f"batched against batch 1: {len(different)} of {len(batched_rows)} batched cells differ; "
          f"h_last cosine >= {batched_rows.h_cosine.min():.5f}, largest entropy gap {batched_rows.entropy_diff.max():.4f}")
    if len(different):
        print(different.to_string())
else:
    print("skipped: RUN_BATCH_CHECK is False")

### Step-2b:- The grid batches

The batches are cut once from the full item list (`ctrl_train` + `dev`) and every cell of the grid uses the same ones.
`batch_grouping: equal_length` puts only items with equal visual and other token counts together (no padding), so a
batch can hold fewer than `GRID_BATCH` items; an item alone runs on the batch-1 path. Read the sizes below.

In [ ]:
items = api.grid_items(grid_cfg, ROOT)
chunks = api.item_chunks(vlm, items, grid_cfg, GRID_BATCH)
sizes = pd.Series([len(chunk) for chunk in chunks], name="batches")
in_full = sizes[sizes == GRID_BATCH].sum() / len(items)
alone = (sizes == 1).sum() / len(items)
print(f"{len(items)} items in {len(chunks)} batches | effective mean batch size {len(items) / len(chunks):.2f} | "
      f"items in a batch of {GRID_BATCH}: {100 * in_full:.0f}% | items alone: {100 * alone:.0f}%")
sizes.value_counts().sort_index().rename_axis("batch size")

#### One batch at the smallest k

The first batch of the largest size, at k = 25% and R = 3: one parsed answer per item, and the batch size each row ran in.

In [ ]:
one_batch = max(chunks, key=len)
K_LOW = min(CHECK["ks"])
print(f"[STAGE] one batch of {len(one_batch)} at k={K_LOW}")
batch_out = api.run_items_cells(vlm, one_batch, {**grid_cfg, "k_grid": [K_LOW], "R_grid": [CHECK["R"]]})
batch_rows = pd.DataFrame([row for rows, _ in batch_out for row in rows])
print("batch sizes used:", batch_rows.batch_size.tolist())
batch_rows[["benchmark", "item_id", "n_vis", "n_input", "batch_size", "pred", "parse_ok"]]

In [ ]:
assert len(batch_rows) == GRID_BATCH, f"no batch of {GRID_BATCH}: the largest has {len(batch_rows)} items"
assert batch_rows.batch_size.eq(GRID_BATCH).all(), f"the batch was split: {batch_rows.batch_size.tolist()}"
assert batch_rows.parse_ok.all(), f"{int((~batch_rows.parse_ok).sum())} answers did not parse"
print(f"✓ one batch of {GRID_BATCH} at k={K_LOW}: {GRID_BATCH} parsed answers")

#### Batch 1 against batch 8

`CHECK["n_items"]` dev items per benchmark at k = 25% and 100% (R = 3), once alone and once in the grid's own batches
(a batch also computes its other items, whose rows are not written). `diff` is batched minus batch 1 in accuracy points
with a paired bootstrap CI; `changed_pct` is the share of parsed answers that differ. A measurement, no pass or fail.
Each regime has its own folder under the grid dir, and a second run continues.

In [ ]:
if RUN_B1_VS_B8:
    check_cfg = {**grid_cfg, "k_grid": CHECK["ks"], "R_grid": [CHECK["R"]]}
    check_items = [it for load in (api.load_gqa, api.load_textvqa)
                   for it in load(cfg, "dev", limit=CHECK["n_items"], root=ROOT)]
    check_dir = OUT_DIR / f"b1_vs_b{GRID_BATCH}"
    regimes = {}
    for size in (1, GRID_BATCH):
        print(f"[STAGE] {len(check_items)} dev items at batch {size}")
        api.run_grid(vlms, items, check_cfg, check_dir / f"b{size}", batch_size=size, only=check_items)
        regimes[size] = api.merge_grid(check_dir / f"b{size}", check_cfg)
else:
    print("skipped: RUN_B1_VS_B8 is False")

In [ ]:
if RUN_B1_VS_B8:
    for size, table in regimes.items():
        assert len(table) == len(check_items) * len(CHECK["ks"]), (size, len(table))
    regime_table = api.batch_regime_table(regimes[1], regimes[GRID_BATCH], CHECK["n_boot"], cfg["splits"]["seed"])
    regime_table.to_csv(OUT_DIR / f"b1_vs_b{GRID_BATCH}.csv", index=False)
    print(f"✓ batch 1 against batch {GRID_BATCH}: {len(regimes[1])} rows each, saved in {OUT_DIR}")
    display(regime_table.round(2))
else:
    print("skipped: RUN_B1_VS_B8 is False")

### Step-3:- A sample through the grid runner

The same function as the full run, writing `N_SAMPLE` dev items (and a re-run of 2 of them) to the sample folder. The
batches are those of the full item list (`only` picks what is written), so the sample runs the batches the grid will run.

In [ ]:
print("[STAGE] sample grid")
sample = api.load_gqa(cfg, "dev", limit=N_SAMPLE // 2, root=ROOT) + api.load_textvqa(cfg, "dev", limit=N_SAMPLE // 2, root=ROOT)
sample_again = api.rerun_items(sample, 2, cfg["splits"]["seed"])
api.run_grid(vlms, items, grid_cfg, SAMPLE_DIR, only=sample)
api.run_grid(vlms, items, grid_cfg, SAMPLE_DIR, rerun_id=1, only=sample_again)
sample_grid = api.merge_grid(SAMPLE_DIR, grid_cfg, params=api.cost_params(vlm.model))
sample_grid.head()

In [ ]:
sample_grid[sample_grid.rerun_id == 0].pivot_table(index="k", columns="R", values="correct", aggfunc="mean").mul(100).round(1)

In [ ]:
sample_grid[sample_grid.rerun_id == 0].pivot_table(index="k", columns="R", values="flops_total", aggfunc="mean").div(1e9).round(2)

In [ ]:
main, again = sample_grid[sample_grid.rerun_id == 0], sample_grid[sample_grid.rerun_id == 1]
assert len(main) == len(sample) * len(CELLS) and len(again) == len(sample_again) * len(CELLS), (len(main), len(again))
assert main.groupby(["benchmark", "item_id"]).size().eq(len(CELLS)).all()
feature_dir = SAMPLE_DIR / "features" / "base" / api.config_hash(grid_cfg)
assert all((feature_dir / f"{it.benchmark}_{it.item_id}.pt").is_file() for it in sample)
pair = main.merge(again, on=["benchmark", "item_id", "k", "R"], suffixes=("", "_again"))
print(f"re-run: {int((pair.pred.fillna('') != pair.pred_again.fillna('')).sum())} of {len(pair)} answers differ")
print(f"✓ sample grid: {len(main)} rows + {len(again)} re-run rows, one features file per item")

### Step-4:- Full run

`RUN_FULL = True` runs every item of the grid splits, then the re-run sample, over the copies of this kernel, and merges
`grid.parquet`. Stop it any time; running the cell again continues: a batch with a missing row is computed again whole,
and the re-run items go through the same batches as in the main run. The same run from a terminal, one shard per GPU group
(free this kernel's models first):

```
python scripts/mvp.py grid --shard 0/2
python scripts/mvp.py grid --shard 1/2
```

In [ ]:
again_items = api.rerun_items(items, cfg["n_rerun"], cfg["splits"]["seed"])
print(len(items), "items +", len(again_items), "re-run | rows expected", (len(items) + len(again_items)) * len(CELLS))

In [ ]:
if RUN_FULL:
    assert FROZEN, "freeze `compressor` in configs/mvp.yaml (S5) before the full run"
    print("[STAGE] full grid")
    api.run_grid(vlms, items, grid_cfg, OUT_DIR)
    api.run_grid(vlms, items, grid_cfg, OUT_DIR, rerun_id=1, only=again_items)
else:
    print("skipped: RUN_FULL is False")

#### Merge and count (also after a terminal run)

In [ ]:
has_results = any(OUT_DIR.glob(f"grid_*_{api.config_hash(grid_cfg)}.jsonl"))
grid = api.merge_grid(OUT_DIR, grid_cfg, params=api.cost_params(vlm.model)) if has_results else None
grid.groupby(["split", "benchmark", "rerun_id"]).size() if has_results else "no full-run files yet"

In [ ]:
if has_results:
    expected = {0: len(items) * len(CELLS), 1: len(again_items) * len(CELLS)}
    got = grid.rerun_id.value_counts().to_dict()
    assert got == expected, f"grid incomplete: rows {got}, expected {expected}"
    assert not (grid.split == "test").any()
    print(f"mean batch size of an item's rows {grid.batch_size.mean():.2f} (batch {GRID_BATCH} asked)")
    print(f"✓ grid.parquet complete: {len(grid)} rows in {OUT_DIR}")
else:
    print("skipped: no full-run files yet")

### Step-5:- Figure F07 and F08: the k x R surface

Reads the saved grid only (no model: the header cells and Settings are enough). F07 is accuracy over k x R, F08 the same
cells as accuracy against FLOPs with the Pareto front. Without a full `grid.parquet` the sample grid is drawn as a draft.

In [ ]:
FINAL = False  # True only after Step-4 printed that grid.parquet is complete: saves to paper/
FIG_SPLITS = ["ctrl_train", "dev"]  # items behind F07 / F08 (main run); test is refused
HAS_GRID = (OUT_DIR / "grid.parquet").is_file()
GRID_FILE = (OUT_DIR if HAS_GRID else SAMPLE_DIR) / "grid.parquet"
assert HAS_GRID or not FINAL, "FINAL needs the full grid.parquet"
saved = api.load_grid(GRID_FILE)
print(GRID_FILE, "|", "full grid" if HAS_GRID else "SAMPLE grid (draft)", "|", len(saved), "rows")
saved.groupby(["split", "benchmark", "rerun_id"]).size()

In [ ]:
surface = api.cell_table(saved, FIG_SPLITS)
surface.round(2)

In [ ]:
api.set_paper_style()
fig = api.plot_kr_accuracy(surface)
api.save_figure(fig, "F07_kr_accuracy", final=FINAL)
fig

In [ ]:
fig = api.plot_cell_pareto(surface)
api.save_figure(fig, "F08_grid_pareto", final=FINAL)
fig

In [ ]:
assert surface.groupby("benchmark").size().eq(len(CELLS)).all(), "a benchmark lacks cells"
print(f"✓ F07 / F08: {len(CELLS)} cells per benchmark, {int(surface.n.min())} items or more per cell")

#### Figure F05: accuracy vs R at 100% tokens, from the grid

The grid's k = 100% cells are the full baseline (same items, R = 1..5), so F05 comes from here instead of a separate nb06 run.


In [ ]:
full_k = saved[(saved.k == 1.0) & (saved.rerun_id == 0) & saved.split.isin(FIG_SPLITS)]
f05 = full_k.groupby(["benchmark", "split", "R"]).agg(n=("correct", "size"), accuracy=("correct", "mean")).reset_index()
f05["accuracy"] *= 100
api.set_paper_style()
fig = api.plot_accuracy_vs_r(f05)
api.save_figure(fig, "F05_accuracy_vs_r", final=FINAL)
f05.round(1)


### Step-6:- Nondeterminism: run 0 against run 1

The `n_rerun` items ran twice with the same settings. How often does the answer flip, and how far does accuracy move?
`accuracy_diff` is run 1 minus run 0 in points.

In [ ]:
noise = api.noise_rate(saved)
noise.round(2)

In [ ]:
total = noise[noise.benchmark == "all"].iloc[0]
print(f"nondeterminism: {total.answer_flip_pct:.2f}% of answers flip, accuracy difference {total.accuracy_diff:+.2f} pts "
      f"({int(total.n_items)} items, {int(total.n_pairs)} item-cell pairs)")
assert total.n_pairs == (saved.rerun_id == 1).sum(), "some re-run rows have no main row"
print("✓ every re-run row has its main row")

### Step-7:- Headroom vs single-cell best_fixed (preliminary; official gate in nb11, hull) (T2, F09)

How much could a per-item choice of (k, R) gain at the same mean FLOPs? Per benchmark and cost cap (a fraction of
`cost_ref`, the mean FLOPs at k = 1, R = 3): **best fixed** is one cell chosen on `ctrl_train` and scored on `dev`; the
**oracle** picks a cell per dev item with the picks' mean FLOPs under the cap (it sees the answers); the **noise
oracle** is what the oracle wins that does not survive the re-run. Headroom = oracle - best fixed - noise oracle.
These are our readings of DEC3 §1 / S14 (see the docstrings of `headroom_table` and `noise_oracle_gain`). Numbers on
`dev` are a preview: the prereg claim is made on test after S12.

In [ ]:
HEADROOM = cfg["headroom"]
HEADLINE, MIN_PTS = HEADROOM["headline_frac"], HEADROOM["min_pts"]
headroom = None
if HAS_GRID:
    headroom = api.headroom_table(saved, cfg)
else:
    print("skipped: headroom needs the full grid (ctrl_train and dev)")
headroom if headroom is None else headroom.round(2).T

In [ ]:
if headroom is not None:
    t2 = headroom[(headroom.cap_frac - HEADLINE).abs() < 1e-9]
    api.save_table(headroom, "T2_headroom_all_caps", final=FINAL)
    api.save_table(t2, "T2_headroom", final=FINAL)
    for row in t2.itertuples():
        print(f"{row.benchmark} at {HEADLINE} x cost_ref: oracle {row.oracle:.1f} - best fixed {row.best_fixed:.1f} "
              f"- noise {row.noise_oracle:.1f} = headroom {row.headroom:.1f} pts "
              f"[{row.headroom_low:.1f}, {row.headroom_high:.1f}] (n={row.n_items}, re-run n={row.n_noise})")
    met = bool((t2.headroom >= MIN_PTS).any())
    print(f"headroom vs single-cell best_fixed (preliminary; official gate in nb11, hull) (>= {MIN_PTS} pts on >= 1 benchmark, dev preview): {'MET' if met else 'NOT met'}")

In [ ]:
if headroom is not None:
    fig = api.plot_headroom(headroom, HEADLINE)
    api.save_figure(fig, "F09_headroom", final=FINAL)
    display(fig)

In [ ]:
if headroom is not None:
    done = headroom.dropna(subset=["best_fixed"])
    assert not done.empty, "no cost cap fits any cell: raise headroom.cap_fracs"
    assert (done.oracle >= done.best_fixed - 1e-9).all(), "the oracle lost to a fixed cell"
    assert (done.oracle >= done.oracle_eligible - 1e-9).all()
    assert (done.oracle_gflops <= done.cap_gflops * (1 + 1e-9)).all()
    assert (done.best_fixed_gflops <= done.cap_gflops * (1 + 1e-9)).all()
    print(f"✓ headroom: {len(done)} benchmark x cap rows, oracle >= best fixed, both within the cap")
else:
    print("skipped: no full grid")